In [ ]:
# Install / upgrade the packages used in this notebook.
# Run this cell if the packages are not already installed.
%pip install -q langchain==1.3.6 langgraph==1.2.11 langchain-openai==1.2.2 langgraph-checkpoint-sqlite pinecone


In [1]:
import importlib.metadata

# List the distribution package names
packages = ["langchain", "langgraph", "langchain-openai", ]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

langchain version: 1.3.6
langgraph version: 1.2.11
langchain-openai version: 1.2.2


# 1. Imports


In [4]:
import os
import sqlite3
import requests
import getpass
import math

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.agents import create_agent
from langchain.tools import tool

from langgraph.checkpoint.sqlite import SqliteSaver

from pinecone import Pinecone, ServerlessSpec



In [6]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
# os.environ["OPENAI_API_KEY"] = "key here" # or uncomment and enter key here

# Initialize the LLM that will power the agent's reasoning
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

response = llm.invoke("Hello! Respond with the word 'Connected' if you can hear me.")  
print("--- Connection Successful ---")
print(response.content)

Enter your OpenAI API key:  ········


--- Connection Successful ---
Connected


# 2. TOOLS

In [7]:
# TOOL 1: Calculator
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a basic math expression, e.g. '2 + 2', and returns the result.
    """
    print("...Running calculator tool")
    try:
        result = eval( expression,{"__builtins__": None}, vars(math),)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


# TOOL 2: Weather
@tool
def get_weather(city: str) -> str:
    """
    Returns the current weather for a given city.
    """
    print("...Running get_weather tool")
    try:
        # Step 1: Convert city → lat/lon
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_res = requests.get(geo_url).json()

        # Check if the results list exists and is not empty
        if "results" not in geo_res or not geo_res["results"]:
            return "City not found"

        # FIX: Ensure you keep the [0] index to pull from the first matching city search result
        lat = geo_res["results"][0]["latitude"]
        lon = geo_res["results"][0]["longitude"]

        # Step 2: Get weather
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_res = requests.get(weather_url).json()

        temp = weather_res["current_weather"]["temperature"]
        wind = weather_res["current_weather"]["windspeed"]

        return f"Temperature: {temp}°C, Wind Speed: {wind} km/h"

    except Exception as e:
        # Better practice: Return or log the specific error string for easier debugging
        return f"Error fetching weather: {e}"


In [8]:
print(calculator.invoke("3*5"))

...Running calculator tool
15


In [9]:
print(get_weather.invoke("Noida"))

...Running get_weather tool
Temperature: 32.9°C, Wind Speed: 10.8 km/h


# 3. Build the agent with tools

In [11]:
# Register both tools so the agent knows they exist and can pick between them
tools = [calculator, get_weather]
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools.  "
    "CRITICAL: Do NOT use markdown, bolding (**), italics (*), or LaTeX formatting like \\( \\) or \\[ \\]."
    "Provide responses in raw, plain text only. Example: Write '1 + 2 * 4 = 9' instead of structural equations."
)

# Create the agent.
# docstring/description, whether to use a tool or just answer directly --
agent = create_agent(
    model=llm,
    tools=tools,

    # Not needed. The agent figures out the tool based on docstring
    system_prompt=SYSTEM_PROMPT  
)

# 4. RUN THE AGENT

## Try
```
input1: What is 1 + 2 * 4 ?

input2: What is the current weather condition in Moscow ?

input3: I am trying to calculate this math expression: one plus two times four ?

input4: I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside ? Is it going going to be nice today ?

In [12]:
# keeps asking until you type 'exit'

while True:
    user_input = input("\nAsk something (or type 'q' to quit): ")
    if user_input.lower() == "q":
        break

    # create_agent's graph expects/returns a list of chat-style messages
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]}
    )

    # The final answer is the content of the last message in the returned state
    final_answer = result["messages"][-1].content
    print("\nFinal Answer:", final_answer)


Ask something (or type 'q' to quit):  What is 1 + 2 * 4 ?


...Running calculator tool

Final Answer: 1 + 2 * 4 = 9



Ask something (or type 'q' to quit):  What is the current weather condition in Moscow ?


...Running get_weather tool

Final Answer: The current weather in Moscow is 12.2°C with a wind speed of 9.5 km/h.



Ask something (or type 'q' to quit):  I am trying to calculate this math expression: one plus two times four ?


...Running calculator tool

Final Answer: The result of the expression one plus two times four is 9.



Ask something (or type 'q' to quit):  I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside ? Is it going going to be nice today ?


...Running get_weather tool

Final Answer: The current temperature in Moscow is 12.2°C with a wind speed of 9.5 km/h. It seems like a mild day, so it could be nice for a visit to the park.



Ask something (or type 'q' to quit):  q


# Sample run
```

Ask something (or type 'q' to quit):  What is 1 + 2 * 4 ?
...Running calculator tool

Final Answer: 1 + 2 * 4 = 9

Ask something (or type 'q' to quit):  What is the current weather condition in Moscow ?
...Running get_weather tool

Final Answer: The current weather in Moscow is 19.3°C with a wind speed of 6.6 km/h.

Ask something (or type 'q' to quit):  I am trying to calculate this math expression: one plus two times four ?
...Running calculator tool

Final Answer: The result of the expression one plus two times four is 9.

Ask something (or type 'q' to quit):  I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside ? Is it going going to be nice today ?
...Running get_weather tool

Final Answer: The current temperature in Moscow is 19.3°C with a wind speed of 6.6 km/h. It seems like a nice day to go to the park!

Ask something (or type 'q' to quit):  q

# 5. Memory

- **Short-term memory:** remembers the ongoing conversation inside one `thread_id`.
- **Long-term memory:** stores important information so it can be retrieved later, even from another conversation/thread.
- In this notebook, **short-term memory uses SQLite through LangGraph's `SqliteSaver` checkpointer**.
- **Long-term memory uses Pinecone** for semantic storage and retrieval.

The two memory types solve different problems:

- **Short-term:** "What did we just talk about?"
- **Long-term:** "What important information do I know about this user from all the  earlier conversations?"

Architecture used in this notebook:

```text
                    AGENT
                      |
             +--------+--------+
             |                 |
       SHORT-TERM          LONG-TERM
          MEMORY              MEMORY
             |                 |
          SQLite             Pinecone
             |                 |
       thread state       durable facts /
       + messages         preferences
```


## Short-term memory - conversation history

- Scoped to one `thread_id`.
- Implemented here with LangGraph's `SqliteSaver`.
- SQLite persists the checkpoints to a local database file.
- The same `thread_id` lets the agent continue the same conversation after another `.invoke()` call.
- A different `thread_id` represents a different conversation.

> **Industry note:** SQLite is useful for demos, local development, and lightweight applications. For larger multi-user production systems, a server database such as PostgreSQL is generally a better fit.


In [13]:
agent_with_no_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)


# Turn 1: mention the city explicitly
chat1 = "My name is Alex. What is 2 + 7 * 3 ?"
result1 = agent_with_no_memory.invoke(
    {"messages": [{"role": "user", "content": chat1}]},
)
print("Turn 1:", result1["messages"][-1].content)

# Turn 2: a follow-up that does NOT repeat the name.
# Without memory, the agent wouldn't know the name of person
chat2 = "What is my name ?"
result2 = agent_with_no_memory.invoke(
    {"messages": [{"role": "user", "content": chat2}]},
)
print("\nTurn 2:", result2["messages"][-1].content)

...Running calculator tool
Turn 1: 2 + 7 * 3 = 23.

Turn 2: I don't have access to personal information about you, including your name. If you'd like to share your name or any other details, feel free to do so!


## Short-term memory - With SQLite

Instead of keeping checkpoints only in RAM, we use a local SQLite database.

The important point is that **SQLite is the storage layer; `SqliteSaver` is the LangGraph checkpointer that knows how to save and restore agent state.**


In [14]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# SHORT-TERM MEMORY:
# SqliteSaver stores LangGraph checkpoints in a local SQLite database.
# This is persistent on disk, unlike InMemorySaver.

SQLITE_DB = "langgraph_short_term_memory.sqlite"

conn = sqlite3.connect(SQLITE_DB, check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

# Every conversation needs a thread_id.
# The thread_id identifies which short-term conversation state to load.
config = {"configurable": {"thread_id": "demo-conversation-1"}}

# Turn 1
chat1 = "My name is Alex. What is 2 + 7 * 3 ?"
result1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": chat1}]},
    config=config,
)
print("Turn 1:", result1["messages"][-1].content)

# Turn 2 - the agent can use the previous turn from SQLite.
chat2 = "What is my name?"
result2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": chat2}]},
    config=config,
)
print("Turn 2:", result2["messages"][-1].content)

print(f"\nSQLite database: {SQLITE_DB}")


...Running calculator tool
Turn 1: The result of 2 + 7 * 3 is 23.

Turn 2: Your name is Alex.


## Long-term memory - Pinecone vector database

- Long-term memory is separate from conversation history.
- Important facts/preferences are converted into embeddings.
- The embeddings and memory text are stored in **Pinecone**.
- Pinecone performs semantic similarity search to retrieve relevant memories.
- Pinecone memory is separate from the agent's `thread_id`, so it can be retrieved from different conversation threads.

Memory flow:

```text
User message
     |
     v
Agent -> save_long_term_memory
     |
     v
OpenAI Embedding
     |
     v
Pinecone

New conversation
     |
     v
Agent -> search_long_term_memory
     |
     v
Pinecone -> Relevant memory -> Agent
```

For this notebook, `text-embedding-3-small` produces 1536-dimensional embeddings, so the Pinecone index is created with dimension `1536`.


In [ ]:
# -----------------------------
# Pinecone long-term memory
# -----------------------------

# Set your Pinecone API key.
# You can also set PINECONE_API_KEY as an environment variable.
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass.getpass("Enter Pinecone API key: ")

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

PINECONE_INDEX_NAME = "agent-long-term-memory"
PINECONE_NAMESPACE = "demo-user-memory"

# text-embedding-3-small -> 1536 dimensions
EMBEDDING_DIMENSION = 1536

# Create the index only if it does not already exist.
if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    print("Created Pinecone index. Waiting for it to become ready...")

# Get the data-plane index client.
index = pc.index(PINECONE_INDEX_NAME)

# OpenAI embeddings are used before data is sent to Pinecone.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Pinecone index:", PINECONE_INDEX_NAME)
print("Pinecone namespace:", PINECONE_NAMESPACE)


In [ ]:
import uuid

@tool
def save_long_term_memory(memory: str) -> str:
    """Save an important user fact or preference to long-term memory in Pinecone."""

    vector = embeddings.embed_query(memory)
    memory_id = str(uuid.uuid4())

    index.upsert(
        vectors=[
            {
                "id": memory_id,
                "values": vector,
                "metadata": {
                    "type": "user_memory",
                    "text": memory,
                },
            }
        ],
        namespace=PINECONE_NAMESPACE,
    )

    print("...Saved to Pinecone long-term memory")
    return f"Saved long-term memory: {memory}"


@tool
def search_long_term_memory(query: str) -> str:
    """Search Pinecone long-term memory for relevant user facts or preferences."""

    query_vector = embeddings.embed_query(query)

    results = index.query(
        vector=query_vector,
        top_k=3,
        include_metadata=True,
        namespace=PINECONE_NAMESPACE,
    )

    matches = results.matches

    if not matches:
        return "No relevant long-term memory found."

    print("...Searching Pinecone long-term memory")

    return "\n".join(
        f"{i}. {match.metadata.get('text', '')}"
        for i, match in enumerate(matches, start=1)
        if match.metadata
    )


In [ ]:
# Test the Pinecone memory directly before giving it to the agent.

save_long_term_memory.invoke(
    "The user prefers practical Python and AI/ML teaching examples."
)

print("\nRetrieved memory:")
print(
    search_long_term_memory.invoke(
        "What teaching style does the user prefer?"
    )
)


## Agent with both short-term and long-term memory

The agent now has two different memory mechanisms:

1. **Short-term memory**
   - `SqliteSaver`
   - SQLite database
   - Conversation history/checkpoints
   - Scoped to a `thread_id`

2. **Long-term memory**
   - Pinecone vector database
   - Stores important facts/preferences as embeddings
   - Retrieved semantically with `search_long_term_memory`
   - Available across different conversation threads

The agent is instructed to search long-term memory when a question may depend on a previously stored fact, and to save durable user preferences/facts when appropriate.


In [ ]:
# Add the Pinecone memory tools to the original calculator + weather tools.

memory_tools = [
    save_long_term_memory,
    search_long_term_memory,
]

tools_with_memory = tools + memory_tools

MEMORY_SYSTEM_PROMPT = SYSTEM_PROMPT + (
    "\nYou also have long-term memory tools backed by a Pinecone vector database. "
    "Use search_long_term_memory when the user's question may depend on information "
    "from an earlier conversation or a stored preference. "
    "Use save_long_term_memory when the user gives an important durable fact, "
    "preference, name, or other information that would be useful in future conversations. "
    "Do not save every message; save only useful long-term information."
)

agent_with_short_and_long_memory = create_agent(
    model=llm,
    tools=tools_with_memory,
    system_prompt=MEMORY_SYSTEM_PROMPT,
    checkpointer=checkpointer,  # SQLite = short-term memory
)


In [ ]:
# IMPORTANT:
# SQLite stores short-term state separately for each thread_id.
# Pinecone stores long-term memory separately from thread_id.
# Therefore, the same Pinecone memory can be retrieved from different threads.

config_user_1 = {"configurable": {"thread_id": "user-1-conversation-1"}}

chat1 = (
    "My name is Alex. Please remember that I prefer Python examples "
    "that are simple and suitable for classroom teaching."
)

result1 = agent_with_short_and_long_memory.invoke(
    {"messages": [{"role": "user", "content": chat1}]},
    config=config_user_1,
)

print("Turn 1:", result1["messages"][-1].content)

chat2 = "What is my name, and what teaching style do I prefer?"

result2 = agent_with_short_and_long_memory.invoke(
    {"messages": [{"role": "user", "content": chat2}]},
    config=config_user_1,
)

print("\nTurn 2:", result2["messages"][-1].content)


In [ ]:
# NEW conversation/thread:
# SQLite short-term conversation history is different because the thread_id changed.
# Pinecone long-term memory is still available.

config_user_1_new_thread = {
    "configurable": {"thread_id": "user-1-conversation-2"}
}

chat3 = "What do you remember about my preferred teaching style?"

result3 = agent_with_short_and_long_memory.invoke(
    {"messages": [{"role": "user", "content": chat3}]},
    config=config_user_1_new_thread,
)

print("New thread:", result3["messages"][-1].content)


## What this demonstrates

| Memory type | Storage | Scope | Example |
|---|---|---|---|
| Short-term memory | `SqliteSaver` + SQLite | One `thread_id` | Previous turns in the current chat |
| Long-term memory | Pinecone vector database | Across threads | User preferences and durable facts |

So:

```text
Same thread
    -> SQLite short-term memory
    -> Pinecone long-term memory

New thread
    -> New SQLite conversation state
    -> Pinecone long-term memory is still available
```

### Practical industry perspective

- **SQLite:** excellent for local development, teaching, demos, and lightweight applications.
- **PostgreSQL:** a stronger choice for a larger production application with many concurrent users.
- **Pinecone:** used here for semantic long-term memory, where similarity search is useful.

The important architectural idea is:

```text
Agent
 |
 +---- Short-term memory ----> LangGraph checkpointer ----> SQLite
 |
 +---- Long-term memory -----> Embeddings ---------------> Pinecone
```

This keeps conversation state and durable semantic memory as two separate systems with different jobs.
